In [1]:
!pip install -q transformers datasets accelerate peft

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

model = model.to(device)

print("Model loaded.")
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Parameters: 494.032768 M


In [4]:
import copy

policy = model
reference = copy.deepcopy(model)

# Policy is trainable
for param in policy.parameters():
    param.requires_grad = True

# Reference is frozen
for param in reference.parameters():
    param.requires_grad = False

reference.eval()

print("Policy trainable:", any(p.requires_grad for p in policy.parameters()))
print("Reference trainable:", any(p.requires_grad for p in reference.parameters()))

Policy trainable: True
Reference trainable: False


In [5]:
max_diff = 0.0

for p1, p2 in zip(policy.parameters(), reference.parameters()):
    diff = (p1.detach() - p2.detach()).abs().max().item()
    max_diff = max(max_diff, diff)

print("Maximum parameter difference:", max_diff)

Maximum parameter difference: 0.0


In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

print(dataset)

Dataset({
    features: ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'],
    num_rows: 61135
})


In [8]:
train_dataset = dataset.select(range(2000))

print("Training examples:", len(train_dataset))

Training examples: 2000


In [9]:
sample = train_dataset[0]

print(sample.keys())

print("\nPROMPT:")
print(sample["prompt"])

print("\nCHOSEN:")
print(sample["chosen"])

print("\nREJECTED:")
print(sample["rejected"])

dict_keys(['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected'])

PROMPT:
how can i develop a habit of drawing daily

CHOSEN:
[{'content': 'how can i develop a habit of drawing daily', 'role': 'user'}, {'content': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Focus on imp

In [10]:
sample = train_dataset[0]

print("PROMPT:")
for msg in sample["prompt"]:
    print(msg)

print("\nCHOSEN:")
for msg in sample["chosen"]:
    print(msg)

print("\nREJECTED:")
for msg in sample["rejected"]:
    print(msg)

PROMPT:
h
o
w
 
c
a
n
 
i
 
d
e
v
e
l
o
p
 
a
 
h
a
b
i
t
 
o
f
 
d
r
a
w
i
n
g
 
d
a
i
l
y

CHOSEN:
{'content': 'how can i develop a habit of drawing daily', 'role': 'user'}
{'content': "Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:\n\n1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.\n2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.\n3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Focus on improving your skills gradually.\n4. Use a variety of tools and medi

In [11]:
prompt_messages = sample["prompt"]
chosen_messages = sample["chosen"]
rejected_messages = sample["rejected"]

chosen_text = tokenizer.apply_chat_template(
    chosen_messages,
    tokenize=False,
    add_generation_prompt=False
)

rejected_text = tokenizer.apply_chat_template(
    rejected_messages,
    tokenize=False,
    add_generation_prompt=False
)

print("CHOSEN TEXT:\n")
print(chosen_text)

print("\n" + "="*80 + "\n")

print("REJECTED TEXT:\n")
print(rejected_text)

CHOSEN TEXT:

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
how can i develop a habit of drawing daily<|im_end|>
<|im_start|>assistant
Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to help you develop the habit of drawing daily:

1. Set a specific time: Allocate a specific time of the day to draw. It could be in the morning, afternoon, or evening. Make drawing a part of your daily routine.
2. Set a specific duration: Determine the amount of time you want to spend on drawing each day. It can be as little as 10 minutes or as long as an hour. Be consistent with the duration to help build the habit.
3. Start small and simple: Don't try to create a masterpiece every day, start with simple and easy-to-do sketches. Focus on improving your skills gradually.
4. Use a variety of too

For DPO, we need:

prompt + chosen
prompt + rejected

But we only calculate the log-probability of the assistant response.

So conceptually:

[user tokens] [assistant tokens]
     ↓              ↓
   MASK=0         MASK=1

The prompt isn't part of the DPO score.

In [12]:
chosen_tokens = tokenizer(
    chosen_text,
    return_tensors="pt"
)

rejected_tokens = tokenizer(
    rejected_text,
    return_tensors="pt"
)

print("Chosen token count:", chosen_tokens["input_ids"].shape[1])
print("Rejected token count:", rejected_tokens["input_ids"].shape[1])

Chosen token count: 381
Rejected token count: 223


In [13]:
with torch.no_grad():
    outputs = policy(
        input_ids=chosen_tokens["input_ids"].to(device),
        attention_mask=chosen_tokens["attention_mask"].to(device)
    )

logits = outputs.logits

print("Input shape:", chosen_tokens["input_ids"].shape)
print("Logits shape:", logits.shape)

Input shape: torch.Size([1, 381])
Logits shape: torch.Size([1, 381, 151936])


means:

1 → batch size
100 → sequence positions
151936 → Qwen's vocabulary size

Suppose our tokens are:

The   cat   sat

At position 0, the model predicts:

cat

At position 1, it predicts:

sat

So:

logits[0] → predicts token 1
logits[1] → predicts token 2
logits[2] → predicts token 3

In [14]:
input_ids = chosen_tokens["input_ids"].to(device)

shifted_logits = logits[:, :-1, :]
shifted_labels = input_ids[:, 1:]

print("Shifted logits:", shifted_logits.shape)
print("Shifted labels:", shifted_labels.shape)

Shifted logits: torch.Size([1, 380, 151936])
Shifted labels: torch.Size([1, 380])


Convert logits → log probabilities

We don't need probabilities themselves.

Use log_softmax:

In [15]:
import torch.nn.functional as F

log_probs = F.log_softmax(shifted_logits, dim=-1)

print("Log-prob shape:", log_probs.shape)

Log-prob shape: torch.Size([1, 380, 151936])


In [16]:
token_log_probs = torch.gather(
    log_probs,
    dim=-1,
    index=shifted_labels.unsqueeze(-1)
).squeeze(-1)

print("Token log-prob shape:", token_log_probs.shape)
print(token_log_probs[0, :10])

Token log-prob shape: torch.Size([1, 380])
tensor([-12.8672,  -1.7236,  -7.8203,  -0.2844, -11.2188,  -1.5820,  -0.8716,
         -6.8164,  -0.1068,  -0.2583], device='cuda:0', dtype=torch.float16)


Now the shape is:

[1, T-1]

Instead of:

[1, T-1, vocab_size]

We've gone from:

every possible token

to:

the probability of the actual token

at each position.

Right now token_log_probs contains probabilities for:

prompt tokens + assistant tokens

But DPO wants:

$$ \boxed{\log P(\text{assistant response}|\text{prompt})} $$

So we need to mask out the prompt.

For example:

Tokens:
[user] [question] [assistant] [answer answer answer]
  0        0          0          1      1      1

Then:

$$ \log\pi(y|x) = \sum_{\text{response tokens}} \log P(y_t|x,y_{<t}) $$
This masking step is where we need to be careful with Qwen's chat template.


Create the response mask

We have:

[user] question [assistant] answer answer answer
  0       0          0        1      1      1

We want:

$$ \log\pi(y|x) = \sum_{\text{answer tokens}} \log P(y_t|x,y_{<t}) $$
Easiest reliable way

Since the dataset gives us the full conversation, we'll identify where the assistant response begins.

For Qwen, let's use the chat template to separately format:

The prompt only
The complete chosen conversation

Then the difference in token lengths tells us where the response starts.

In [17]:
prompt_text = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True
)

chosen_text = tokenizer.apply_chat_template(
    chosen_messages,
    tokenize=False,
    add_generation_prompt=False
)

prompt_ids = tokenizer(
    prompt_text,
    add_special_tokens=False
)["input_ids"]

chosen_ids = tokenizer(
    chosen_text,
    add_special_tokens=False
)["input_ids"]

print("Prompt tokens:", len(prompt_ids))
print("Chosen total tokens:", len(chosen_ids))
print("Response tokens:", len(chosen_ids) - len(prompt_ids))

Prompt tokens: 24
Chosen total tokens: 381
Response tokens: 357


In [18]:
response_start = len(prompt_ids)

response_mask = torch.zeros(
    len(chosen_ids) - 1,
    dtype=torch.bool
)

response_mask[response_start - 1:] = True

print("Total scored positions:", response_mask.sum().item())
print("Response token count:", len(chosen_ids) - response_start)

Total scored positions: 357
Response token count: 357


Calculate the actual sequence log-probability

Now combine the pieces we already built.

Now we have:

$$ \boxed{ \log\pi_\theta(y_w|x) } $$

for the chosen response.

In [19]:
selected_log_probs = token_log_probs[0][response_mask]

sequence_log_prob = selected_log_probs.sum()

print("Chosen log probability:", sequence_log_prob.item())

Chosen log probability: -507.5


unction to calculate logπ(y∣x)

In [20]:
def get_sequence_logprob(model, messages):
    """
    Computes log P(assistant response | conversation prompt).

    Only assistant-response tokens contribute.
    """

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Conversation up to the point where assistant should answer
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    full = tokenizer(
        full_text,
        return_tensors="pt",
        add_special_tokens=False
    )

    prompt = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False
    )

    input_ids = full["input_ids"].to(device)
    attention_mask = full["attention_mask"].to(device)

    prompt_len = prompt["input_ids"].shape[1]

    # Forward pass
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    # Shift for next-token prediction
    logits = outputs.logits[:, :-1, :]
    labels = input_ids[:, 1:]

    # Log probabilities
    log_probs = F.log_softmax(logits, dim=-1)

    # Pick probability of actual next token
    token_log_probs = torch.gather(
        log_probs,
        dim=-1,
        index=labels.unsqueeze(-1)
    ).squeeze(-1)

    # Only score assistant response
    response_mask = torch.zeros_like(token_log_probs, dtype=torch.bool)
    response_mask[:, prompt_len - 1:] = True

    sequence_logprob = (
        token_log_probs * response_mask
    ).sum(dim=1)

    return sequence_logprob

In [21]:
sample = train_dataset[0]

chosen_logprob = get_sequence_logprob(
    policy,
    sample["chosen"]
)

rejected_logprob = get_sequence_logprob(
    policy,
    sample["rejected"]
)

print("Chosen log P:", chosen_logprob.item())
print("Rejected log P:", rejected_logprob.item())

Chosen log P: -440.75
Rejected log P: -243.125


Get the four numbers DPO needs
logπ
θ
	​

(y
w
	​

∣x)
	​

$$ \boxed{ \log\pi_\theta(y_l|x) } $$ $$ \boxed{ \log\pi_{\rm ref}(y_w|x) } $$
logπ
ref
	​

(y
l
	​

∣x)
	​


                   SAME DATA
                       │
            ┌──────────┴──────────┐
            ↓                     ↓
       CHOSEN ANSWER         REJECTED ANSWER
            │                     │
       ┌────┴────┐           ┌────┴────┐
       ↓         ↓           ↓         ↓
      πθ       πref         πθ       πref
       │         │           │         │
       ↓         ↓           ↓         ↓
     -8.2      -9.1        -11.4     -10.7

In [22]:
chosen_policy = get_sequence_logprob(
    policy,
    sample["chosen"]
)

rejected_policy = get_sequence_logprob(
    policy,
    sample["rejected"]
)

with torch.no_grad():
    chosen_reference = get_sequence_logprob(
        reference,
        sample["chosen"]
    )

    rejected_reference = get_sequence_logprob(
        reference,
        sample["rejected"]
    )

print("πθ chosen:  ", chosen_policy.item())
print("πθ rejected: ", rejected_policy.item())

print("πref chosen: ", chosen_reference.item())
print("πref rejected:", rejected_reference.item())

πθ chosen:   -440.75
πθ rejected:  -243.125
πref chosen:  -440.75
πref rejected: -243.125
